# DistilBERT 预训练模型扩展

在同一份清洗后的 IMDB 训练集和验证集上微调英文 DistilBERT。使用固定版本的官方基础权重，初始化二分类头，并更新全部模型参数。性能差异同时受到预训练和分词方式的影响。

模型来源为 [DistilBERT 基础模型](https://huggingface.co/distilbert/distilbert-base-uncased/tree/12040accade4e8a0f71eabdb258fecc2e7e948be)。下载该版本的 config.json、tokenizer.json、tokenizer_config.json、vocab.txt 和 model.safetensors 到 data/pretrained/distilbert-base-uncased。权重和训练输出均保存在被 Git 忽略的 data 目录。

本次预设最大输入长度为 256（含特殊词项）、种子 42、两轮微调，按最低验证损失选模型。输入长度统计用来记录截断情况，当前 Notebook 只读取训练集和验证集。


In [ ]:
import hashlib
import json
import math
import time

import pandas as pd
import torch
import transformers
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from transformers import get_linear_schedule_with_warmup

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.paths import CLEANED_DIR, CONFIGS_DIR, DATA_DIR, TABLES_DIR

model_dir = DATA_DIR / "pretrained" / "distilbert-base-uncased"
output_dir = DATA_DIR / "models" / "distilbert_seed42"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
precision = "bf16" if device.type == "cuda" and torch.cuda.is_bf16_supported() else "fp32"

config = {
    "model_id": "distilbert/distilbert-base-uncased",
    "revision": "12040accade4e8a0f71eabdb258fecc2e7e948be",
    "seed": 42,
    "max_length": 256,
    "batch_size": 16,
    "gradient_accumulation_steps": 2,
    "epochs": 2,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "max_gradient_norm": 1.0,
    "selection": "minimum validation cross-entropy loss",
    "precision": precision,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "torch": str(torch.__version__),
    "transformers": transformers.__version__,
}

paths = {"train": CLEANED_DIR / "train.csv", "validation": CLEANED_DIR / "val.csv"}
data = {name: pd.read_csv(path) for name, path in paths.items()}
for name, split in data.items():
    if split.empty or split["text"].isna().any() or not split["label"].isin([0, 1]).all():
        raise ValueError(f"Invalid data in {name}.")
config["split_sha256"] = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in paths.items()}
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, trust_remote_code=False)
config["model_sha256"] = {
    filename: hashlib.sha256((model_dir / filename).read_bytes()).hexdigest()
    for filename in ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json", "vocab.txt"]
}
print(f"Device: {config['device']} | Precision: {precision}")


In [ ]:
# 按模型 tokenizer 统计输入长度，并准备动态填充的批次
coverage = []
encoded = {}
for name, split in data.items():
    lengths = []
    samples = []
    for start in range(0, len(split), 512):
        chunk = split.iloc[start:start+512]
        texts = chunk["text"].tolist()
        original = tokenizer(texts, truncation=False, return_length=True, verbose=False)
        lengths.extend(original["length"])
        features = tokenizer(texts, truncation=True, max_length=config["max_length"])
        samples.extend({"input_ids": ids, "attention_mask": mask, "labels": int(label)}
                       for ids, mask, label in zip(features["input_ids"], features["attention_mask"], chunk["label"]))
    encoded[name] = samples
    for limit in [128, 256, 512]:
        truncated = sum(length > limit for length in lengths)
        coverage.append({"SPLIT": name, "MAX LENGTH": limit, "ROWS": len(lengths),
                         "TRUNCATED": truncated, "COVERAGE": 1 - truncated / len(lengths)})

TABLES_DIR.mkdir(parents=True, exist_ok=True)
coverage_table = pd.DataFrame(coverage)
display(coverage_table.round(4))
coverage_table.to_csv(TABLES_DIR / "distilbert_token_coverage.csv", index=False)

collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
generator = torch.Generator().manual_seed(config["seed"])
train_loader = DataLoader(encoded["train"], batch_size=config["batch_size"], shuffle=True,
                          generator=generator, collate_fn=collator, pin_memory=device.type == "cuda")
validation_loader = DataLoader(encoded["validation"], batch_size=config["batch_size"], shuffle=False,
                               collate_fn=collator, pin_memory=device.type == "cuda")


In [ ]:
# 加载基础权重并固定微调配置
torch.manual_seed(config["seed"])
model = AutoModelForSequenceClassification.from_pretrained(
    model_dir, num_labels=2, local_files_only=True, use_safetensors=True, trust_remote_code=False,
    id2label={0: "NEGATIVE", 1: "POSITIVE"}, label2id={"NEGATIVE": 0, "POSITIVE": 1},
).to(device)
config["trainable_parameters"] = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
config["attention_implementation"] = model.config._attn_implementation
protocol_path = CONFIGS_DIR / "distilbert_finetuning.json"
if protocol_path.exists() or output_dir.exists():
    raise FileExistsError("A DistilBERT run already exists. Inspect its artifacts before starting a new run.")
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
protocol_path.write_text(json.dumps(config, ensure_ascii=False, indent=2) + chr(10), encoding="utf-8")
protocol_hash = hashlib.sha256(protocol_path.read_bytes()).hexdigest()

optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
accumulation = config["gradient_accumulation_steps"]
total_updates = config["epochs"] * math.ceil(len(train_loader) / accumulation)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(config["warmup_ratio"] * total_updates), num_training_steps=total_updates,
)
print(f"Fixed protocol: {protocol_hash} | Updates: {total_updates:,}")


In [ ]:
def evaluate(model, loader):
    model.eval()
    loss_sum = 0.0
    labels = []
    predictions = []
    with torch.inference_mode():
        for batch in loader:
            batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=precision == "bf16"):
                output = model(**batch)
            loss_sum += float(output.loss) * len(batch["labels"])
            labels.extend(batch["labels"].cpu().tolist())
            predictions.extend(output.logits.argmax(dim=1).cpu().tolist())
    if labels != data["validation"]["label"].tolist():
        raise ValueError("Validation order or labels do not match.")
    if not math.isfinite(loss_sum):
        raise ValueError("Validation loss is not finite.")
    return loss_sum / len(labels), classification_metrics(labels, predictions)


history = []
best_loss = float("inf")
updates = 0
for epoch in range(1, config["epochs"] + 1):
    started = time.perf_counter()
    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_loss = 0.0
    for step, batch in enumerate(train_loader, start=1):
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=precision == "bf16"):
            output = model(**batch)
        batch_loss = float(output.loss.detach())
        if not math.isfinite(batch_loss):
            raise ValueError("Training loss is not finite.")
        batch_rows = len(batch["labels"])
        train_loss += batch_loss * batch_rows
        group_start = ((step - 1) // accumulation) * accumulation * config["batch_size"]
        group_rows = min(accumulation * config["batch_size"], len(encoded["train"]) - group_start)
        (output.loss * (batch_rows / group_rows)).backward()

        if step % accumulation == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_gradient_norm"])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            updates += 1
        if step % 200 == 0:
            print(f"Epoch {epoch} | batch {step:,}/{len(train_loader):,} | loss {batch_loss:.4f}", flush=True)

    validation_loss, metrics = evaluate(model, validation_loader)
    row = {"EPOCH": epoch, "TRAIN LOSS": train_loss / len(encoded["train"]),
           "VALIDATION LOSS": validation_loss, **metrics, "SECONDS": time.perf_counter() - started}
    print(row, flush=True)
    history.append(row)
    if validation_loss < best_loss:
        best_loss = validation_loss
        best_result = row
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
        (output_dir / "run_metadata.json").write_text(
            json.dumps({"protocol_sha256": protocol_hash, "best_epoch": epoch}, indent=2) + chr(10),
            encoding="utf-8",
        )
    pd.DataFrame(history).to_csv(TABLES_DIR / "distilbert_training_history.csv", index=False)

if updates != total_updates:
    raise ValueError("Optimizer update count does not match the fixed configuration.")


In [ ]:
# 恢复最低验证损失的模型并与既有结果比较
del optimizer, scheduler, model
best_model = AutoModelForSequenceClassification.from_pretrained(
    output_dir, local_files_only=True, use_safetensors=True, trust_remote_code=False,
).to(device)
verified_loss, verified_metrics = evaluate(best_model, validation_loader)
if not math.isclose(verified_loss, best_loss, abs_tol=1e-6) or any(
    not math.isclose(value, best_result[metric], abs_tol=1e-12) for metric, value in verified_metrics.items()
):
    raise ValueError("Saved checkpoint does not reproduce validation metrics.")
fixed_summary = pd.read_csv(TABLES_DIR / "fixed_experiment_summary.csv")
metric_names = ["ACCURACY", "PRECISION", "RECALL", "F1"]
comparison = fixed_summary[["MODEL"] + ["VALIDATION " + name for name in metric_names]].copy()
comparison.loc[len(comparison)] = ["DistilBERT"] + [verified_metrics[name] for name in metric_names]
comparison.to_csv(TABLES_DIR / "distilbert_validation_comparison.csv", index=False)
display(comparison.round(4))
print(f"Best checkpoint reloaded and verified: epoch {best_result['EPOCH']}")


## 本次结果

第一轮验证损失为 0.2290，F1 为 0.9088。第二轮 F1 为 0.9147，验证损失升至 0.2364。按预先确定的最低验证损失规则，保留第一轮。重新加载检查点后，验证指标与第一轮记录一致。

使用 256 个词项时，验证集 1,027 条评论被截断，占 41.23%。在同一验证集上，DistilBERT 的 F1 为 0.9088，B06 为 0.8889，CNN + 单向 LSTM 等权融合为 0.8828。比较使用了一个随机种子，模型的预训练资源、分词和输入长度各不相同。

本次微调和指标比较只使用训练集及验证集。后续测试应先固定此检查点和评估设置，并说明项目中其他方案的测试结果已经查看。
